In [151]:
# conda activate genomic_tools

import os
import json
import pickle
import pandas as pd
from collections import defaultdict

pd.set_option('display.max_columns', None)

## Load interproscan results

In [148]:
cols = [
    'protein_accession',
    'sequence_md5',
    'sequence_length',
    'analysis',
    'signature_accession',
    'signature_description',
    'start',
    'stop',
    'score',
    'status',
    'date',
    'interpro_accession',
    'interpro_description',
    'go_annotations',
    'pathways'
]

interproscan_results = pd.read_csv(
    "data/interproscan/results/proteins.fa.tsv",
    sep='\t',
    header=None,
    names=cols,
    index_col=False
)

In [149]:
# Parse the PIRSR data

with open("data/interproscan/interpro/data/pirsr/sr_uru.json") as f:
    pirsr_data = json.load(f)

# subset to proteins patterns applicable to humans
human_relevant = ['Eukaryota', 'Eukaryota; Metazoa', 'Eukaryota; Vertebrata', 'Eukaryota; Chordata', 'Eukaryota; Mammalia', 'Eukaryota; Eutheria']

records = []
for ac, entry in pirsr_data.items():
    for group_id, sites in entry['Groups'].items():
        for site in sites:
            scope = entry.get('Scope', [])
            tr = entry.get('TR', '')
            if any(s in human_relevant for s in scope):
                records.append({
                    'accession': ac,
                    'scope':  ', '.join(scope),
                    'TR': tr.split("; ")[1],
                    'label': site['label'],
                    'condition': site['condition'],
                    'desc': site['desc'],
                    'group': group_id
                })

pirsr_df = pd.DataFrame(records)

pirsr_df = pirsr_df.groupby("accession").agg(
    scope=('scope', lambda x: ' | '.join(x.unique())),
    TR=('TR', lambda x: ' | '.join(x.unique())),
    label=('label', lambda x: ' | '.join(x.unique())),
    condition=('condition', lambda x: ' | '.join(x.unique())),
    desc=('desc', lambda x: ' | '.join(x.unique())),
    group=('group', lambda x: ' | '.join(x.unique())),
).reset_index()

In [150]:
interproscan_results = interproscan_results.merge(pirsr_df, left_on="signature_accession", right_on="accession", how="left")

interproscan_results.loc[interproscan_results['analysis'] == "PIRSR", 'signature_description'] = interproscan_results.loc[interproscan_results['analysis'] == "PIRSR", 'label'] 

## Map events to interproscan results

(Get the transcript associated with each significant event)

In [6]:
with open("data/gencode.v46.annotation_cds_by_transcript.pkl", "rb") as f:
    cds_by_transcript = pickle.load(f)

In [153]:
with open('data/event_protein_map.pkl', 'rb') as f:
    event_protein_map = pickle.load(f)

In [246]:
rec_incl['exon_cds_start']

999692

In [ ]:
# def get_skip_junction_aa(skip_transcript, exon_cds_start, exon_cds_end, cds_by_transcript):
#     cds = _cds_rows(cds_by_transcript.get(skip_transcript))
#     if cds is None:
#         return None
#     nt_before = 0
#     for c in cds:
#         # stop when we reach or pass the cassette exon's position
#         if c['start'] >= exon_cds_start:
#             break
#         # only count rows that end before the cassette exon starts
#         if c['end'] < exon_cds_start:
#             nt_before += c['end'] - c['start'] + 1
#         else:
#             # partial overlap: count only up to exon_cds_start
#             nt_before += exon_cds_start - c['start']
#             break
#     return nt_before // 3

def near_exon(exon_start, exon_end, scan_start, scan_end, window=300):
    """Boolean: features are within window (nts) of skipped exon."""
    return (abs(scan_start - exon_start) < window) | \
        (abs(scan_end - exon_end) < window)
    
def _cds_rows(obj):
    if obj is None:
        return None
    if hasattr(obj, "iterrows"):
        return [{'start': int(r['start']), 'end': int(r['end']), 'frame': int(r['frame'])}
                for _, r in obj.iterrows()]
    return [{'start': int(c['start']), 'end': int(c['end']), 'frame': int(c['frame'])} for c in obj]

def rel_cds_to_genome(cds_obj, strand):
    """
    Build a list of exon segments mapping between genomic coordinates
    and CDS-relative coordinates, in transcript (5'->3') order.

    Each segment is a dict:
        genome_start, genome_end : genomic coordinates (always genome_start <= genome_end)
        rel_start, rel_end       : CDS-relative coordinates (always rel_start <= rel_end)
    """
    cds = _cds_rows(cds_obj)

    # order exons in transcript (5'->3') order
    if strand == '-':
        cds = sorted(cds, key=lambda c: c['start'], reverse=True)
    else:
        cds = sorted(cds, key=lambda c: c['start'])

    segments = []
    cds_start = 0
    for c in cds:
        length = c['end'] - c['start'] + 1
        segments.append({
            'genome_start': c['start'],
            'genome_end': c['end'],
            'rel_start': cds_start,
            'rel_end': cds_start + length - 1,
        })
        cds_start += length

    return segments


def map_aa_to_genome(rel_to_genome, seq_start, seq_end):
    """
    Given a CDS-relative range [seq_start, seq_end], return the
    corresponding genomic interval(s) as a list of (genome_start, genome_end)
    tuples -- one entry per exon the range touches, in genomic order.
    """
    result = []
    for seg in rel_to_genome:
        overlap_start = max(seq_start, seg['rel_start'])
        overlap_end = min(seq_end,  seg['rel_end'])

        if overlap_start <= overlap_end:
            # offset from rel_start maps directly onto genome_start,
            # regardless of strand, since genome_start<=genome_end and
            # rel_start<=rel_end always hold by construction
            g_start = seg['genome_start'] + (overlap_start - seg['rel_start'])
            g_end   = seg['genome_start'] + (overlap_end   - seg['rel_start'])
            result.append((g_start, g_end))

    return result

In [ ]:
columns = ['protein_accession', 'sequence_length', 'analysis', 
           'signature_description', 'start', 'stop', 'interpro_description']
analyses_to_exclude = ['NCBIFAM', 'SFLD']
ipr = interproscan_results.loc[~interproscan_results['analysis'].isin(analyses_to_exclude), columns]

ipr_grouped = {acc: grp for acc, grp in ipr.groupby('protein_accession')}

event_interproscan_map = defaultdict(dict)

for ev, rec in event_protein_map.items():
    
    rec_incl = rec['inclusion']
    incl_df  = ipr_grouped.get(rec_incl['transcript_id'])
    if incl_df is None:
        continue
    overlap_df = incl_df[
        (incl_df['start'] <= rec_incl['aa_end']) &
        (incl_df['stop']  >= rec_incl['aa_start'])
    ]
    if overlap_df.empty:
        continue
    
    incl_cols = ['aa_start', 'aa_end', 'exon_cds_start', 'exon_cds_end',
                 'frame_preserving', 'clean_start', 'clean_end']
    
    event_interproscan_map[ev] = {
        'meta': rec['meta'],
        'inclusion': overlap_df.assign(
            meta=rec['meta'],
            **{col: rec_incl[col] for col in incl_cols}
        ).reset_index(drop=True),
        'real_skip': None,
        'synethic_skip': None
    }
    
    # real skip
    if rec.get('real_skip'):
        skip_df = ipr_grouped.get(rec['real_skip'])
        if skip_df is not None: 
            cds_obj = cds_by_transcript[rec['real_skip']]
            cds_to_genome = rel_cds_to_genome(cds_obj, rec['meta']['strand'])
            genome_coords = []
            for idx, row in skip_df.iterrows():
                aa_to_genome = map_aa_to_genome(cds_to_genome, row['start'], row['stop'])
                scan_start = min(min(t) for t in aa_to_genome)
                scan_end = max(max(t) for t in aa_to_genome)
                if near_exon(rec_incl['exon_cds_start'], rec_incl['exon_cds_end'], scan_start, scan_end): 
                    genome_coords.append(aa_to_genome)
                else:
                    genome_coords.append(None)
                    
            skip_df = skip_df.assign(genome_coords=genome_coords)
            skip_df = skip_df[skip_df['genome_coords'].notna()]
                
        if not skip_df.empty:
            event_interproscan_map[ev]['real_skip'] = skip_df.reset_index(drop=True)
                
    # synthetic skip
    if rec.get('synthetic_skip'):
        synth_df = ipr_grouped.get(rec['synthetic_skip'])
        if synth_df is not None:
            cds_obj = cds_by_transcript[rec['inclusion']['transcript_id']]
            cds_to_genome = rel_cds_to_genome(cds_obj, rec['meta']['strand'])
            genome_coords = []
            for idx, row in synth_df.iterrows():
                aa_to_genome = map_aa_to_genome(cds_to_genome, row['start'], row['stop'])
                scan_start = min(min(t) for t in aa_to_genome)
                scan_end = max(max(t) for t in aa_to_genome)
                if near_exon(rec_incl['exon_cds_start'], rec_incl['exon_cds_end'], scan_start, scan_end): 
                    genome_coords.append(aa_to_genome)
                else:
                    genome_coords.append(None)
                    
            synth_df = synth_df.assign(genome_coords=genome_coords)
            synth_df = synth_df[synth_df['genome_coords'].notna()]
            
            if not synth_df.empty:
                event_interproscan_map[ev]['synthetic_skip'] = synth_df.reset_index(drop=True)
            
    # junction siblings
    if rec.get('exon_diff_junction_siblings'):
        frames = []
        for sib in rec['exon_diff_junction_siblings']:
            t = sib['transcript_id']
            if t not in ipr_grouped:
                continue
            sib_df = ipr_grouped[t]
            sib_overlap = sib_df[
                (sib_df['start'] <= sib['aa_end']) &
                (sib_df['stop']  >= sib['aa_start'])
            ].assign(
                aa_start = sib['aa_start'],
                aa_end = sib['aa_end'],
                exon_cds_start = sib['exon_cds_start'],
                exon_cds_end = sib['exon_cds_end'],
            )
            if not sib_overlap.empty:
                frames.append(sib_overlap)
        if frames:
            event_interproscan_map[ev]['exon_diff_junction_siblings'] = pd.concat(
                frames, ignore_index=True
            )

    # boundary siblings
    if rec.get('exon_diff_boundary_siblings'):
        frames = []
        for sib in rec['exon_diff_boundary_siblings']:
            t = sib['transcript_id']
            if t not in ipr_grouped:
                continue
            sib_df = ipr_grouped[t]
            sib_overlap = sib_df[
                (sib_df['start'] <= sib['aa_end']) &
                (sib_df['stop']  >= sib['aa_start'])
            ].assign(
                aa_start = sib['aa_start'],
                aa_end = sib['aa_end'],
                exon_cds_start = sib['exon_cds_start'],
                exon_cds_end = sib['exon_cds_end'],
            )
            if not sib_overlap.empty:
                frames.append(sib_overlap)
        if frames:
            event_interproscan_map[ev]['exon_diff_boundary_siblings'] = pd.concat(
                frames, ignore_index=True
            )

In [302]:
aa_to_genome

[(999944, 999973), (999526, 999529)]

In [301]:
cds_obj['start'] != rec_incl['exon_cds_start']

1605     True
1608    False
1610     True
1612     True
Name: start, dtype: bool

In [300]:
cds_obj[cds_obj['start'] < rec_incl['exon_cds_start']]

,chrom,start,end,feature,strand,transcript,gene_type,gene,transcript_type,exon_id,exon_number,frame,tag,protein_id
1610,chr1,999526,999613,CDS,-,ENST00000304952,protein_coding,HES4,protein_coding,ENSE00001801835,3,0,"basic,Ensembl_canonical,GENCODE_Primary,MANE_S...",ENSP00000304595.7
1612,chr1,999062,999432,CDS,-,ENST00000304952,protein_coding,HES4,protein_coding,ENSE00001934557,4,2,"basic,Ensembl_canonical,GENCODE_Primary,MANE_S...",ENSP00000304595.7


In [ ]:
rec

In [ ]:
frame_preserving = rec_incl['frame_preserving']

[{'genome_start': 999866,
  'genome_end': 999973,
  'rel_start': 0,
  'rel_end': 107},
 {'genome_start': 999692,
  'genome_end': 999787,
  'rel_start': 108,
  'rel_end': 203},
 {'genome_start': 999526,
  'genome_end': 999613,
  'rel_start': 204,
  'rel_end': 291},
 {'genome_start': 999062,
  'genome_end': 999432,
  'rel_start': 292,
  'rel_end': 662}]

In [9]:
with open('data/event_interproscan_map.pkl', "wb") as file:
    pickle.dump(event_interproscan_map, file)

## Merge interproscan results with significant splicing event info.

In [10]:
with open("data/event_interproscan_map.pkl", "rb") as file:
    event_interproscan_map = pickle.load(file)

In [11]:
# merge cell type-specific events with InterProScan results

signif_event_interproscan_map = dict() 
signif_event_interproscan_summary = dict()

for file in os.listdir("data/ctype_exons"):
    if file.endswith("exons.csv"):
        ctype = file.split("_exons.csv")[0]
        print(ctype)
        
        signif_events_df = pd.read_csv(f"data/ctype_exons/{file}", index_col=0)
        
        event_interproscan_dict = {
            ev: event_interproscan_map[ev] for ev in signif_events_df.index 
            if ev in event_interproscan_map
        }

        result = pd.concat(
            [df.assign(event_id=ev, bucket=bucket)
             for ev, buckets in event_interproscan_dict.items()
             for bucket, df in buckets.items()],
            ignore_index=True
        )
        # move event and bucket to front
        cols = ['event_id', 'bucket'] + [c for c in result.columns if c not in ('event_id', 'bucket')]
        result = result[cols]
 
        # restrict to splicing events with InterProScan results
        df = result.merge(signif_events_df, left_on="event_id", right_index=True)
        signif_event_interproscan_map[ctype] = df
        
        # summarize interpro results for significant splicing events
        signif_event_interproscan_summary[ctype] = df.groupby(["event_id", "bucket"]).agg(
            r=('r', lambda x: ' | '.join(map(str, x.unique()))),
            is_specific=('is_specific', lambda x: ' | '.join(map(str, x.unique()))),
            Gene=('Gene', lambda x: ' | '.join(x.unique())),
            interproscan_id=('protein_accession', lambda x: ' | '.join(x.unique())),
            chr=('chr', lambda x: ' | '.join(x.unique())),
            exon_start=('exon_start', lambda x: ' | '.join(map(str, x.unique()))),
            exon_end=('exon_end', lambda x: ' | '.join(map(str, x.unique()))),
            exon_len=('exon_len', lambda x: ' | '.join(map(str, x.unique()))),
            exon_cds_start=('exon_cds_start', lambda x: ' | '.join(map(str, x.unique()))),
            exon_cds_end=('exon_cds_end', lambda x: ' | '.join(map(str, x.unique()))),
            exon_aa_start=('aa_start', lambda x: ' | '.join(map(str, x.unique()))),
            exon_aa_end=('aa_end', lambda x: ' | '.join(map(str, x.unique()))),
            domain_start=('start', lambda x: ' | '.join(map(str, x.unique()))),
            domain_stop=('stop', lambda x: ' | '.join(map(str, x.unique()))),
            protein_sequence_length=('sequence_length', lambda x: ' | '.join(map(str, x.unique()))),
            frame_preserving=('frame_preserving', lambda x: ' | '.join(map(str, x.unique()))),
            n_analyses=('analysis', lambda x: len(x.unique())),
            analyses=('analysis', lambda x: ' | '.join(x.unique())),
            signature_descriptions=('signature_description', lambda x: ' | '.join(x.unique())),
            interpro_descriptions=('interpro_description', lambda x: ' | '.join(x.unique()))
        ).reset_index()

Oligo


TypeError: sequence item 7: expected str instance, float found

In [ ]:
with open("data/signif_event_interproscan_map.pkl", "wb") as file:
    pickle.dump(signif_event_interproscan_map, file)
    
with open("data/signif_event_interproscan_summary.pkl", "wb") as file:
    pickle.dump(signif_event_interproscan_summary, file)

Preview

In [ ]:
rec = signif_event_interproscan_summary['Deep_layer_glutamatergic']

rec[(rec['is_specific'] == "True") & (rec['bucket'] == "inclusion")].sort_values('n_analyses', ascending=False).head(10)

NameError: name 'signif_event_interproscan_summary' is not defined